# Experiment-1: **Original Images**

In [ ]:
import os
import cv2
import torch
import wandb
import random
import numpy as np
import torch.nn as nn
from glob import glob
import torch.optim as optim
from torch.utils.data import DataLoader, TensorDataset, Subset, ConcatDataset
from torchvision import datasets, transforms, models
from sklearn.model_selection import KFold
from google.colab import userdata
from tqdm.auto import tqdm
from sklearn.metrics import (
    confusion_matrix, classification_report,
    accuracy_score, precision_score, recall_score, f1_score
)
from datetime import datetime, date
today = date.today()
curr_date = today.strftime("%d-%m-%Y")
from torch.utils.data import Dataset, DataLoader, Subset
from PIL import Image
from sklearn.model_selection import KFold
import matplotlib.pyplot as plt
from tqdm import tqdm
import seaborn as sns
import torch.nn.functional as F
from torchvision import models, transforms

In [ ]:
# os.environ["WANDB_API_KEY"] = userdata.get("WANDB_API")
# wandb.login()


/usr/local/lib/python3.12/dist-packages/notebook/notebookapp.py:191: SyntaxWarning: invalid escape sequence '\/'
  | |_| | '_ \/ _` / _` |  _/ -_)
wandb: Currently logged in as: 2310080019 (lkx100-kl-university) to https://api.wandb.ai. Use `wandb login --relogin` to force relogin


True

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


In [ ]:
RANDOM_STATE = 42

# Ensure deterministic behavior
torch.backends.cudnn.deterministic = True
random.seed(hash("setting random seeds") % 2**32 - 1)
np.random.seed(hash("improves reproducibility") % 2**32 - 1)
torch.manual_seed(hash("by removing stochasticity") % 2**32 - 1)
torch.cuda.manual_seed_all(hash("so runs are repeatable") % 2**32 - 1)

# Device configuration
device = torch.device("cuda:0" if torch.cuda.is_available() else "cpu")

In [ ]:

# ---------------- CONFIG ---------------- #
CONFIG = {
    "epochs": 50,
    "batch_size": 16,
    "lr": 1e-4,
    "k_folds": 5,
    "patience": 9,
    # "img_size": 224,
    "num_classes": 3,
    "group_name":"Original",
    "dataset_path": "/content/drive/MyDrive/Dataset/castor_v2_224x224",
    "usewandb":False
}

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")


In [ ]:

# ---------------- DATASET ---------------- #
class PestDataset(Dataset):
    def __init__(self, files, labels, transform=None):
        self.files = files
        self.labels = labels
        self.transform = transform

    def __len__(self):
        return len(self.files)

    def __getitem__(self, idx):
        img = Image.open(self.files[idx]).convert("RGB")
        if self.transform:
            img = self.transform(img)
        return img, self.labels[idx]


# ---------------- LOAD FILES ---------------- #
def load_dataset(root):
    class_folders = sorted(os.listdir(root))
    files, labels = [], []

    for label, cls in enumerate(class_folders):
        img_files = glob(os.path.join(root, cls, "*"))
        files.extend(img_files)
        labels.extend([label] * len(img_files))

    return files, labels


# ---------------- MODEL ---------------- #
def get_model(num_classes=3):
    model = models.mobilenet_v2(pretrained=True)
    # Correctly access in_features from the last Linear layer in the classifier
    model.classifier = nn.Linear(model.classifier[1].in_features, num_classes)
    return model


# ---------------- CLASS WEIGHTS ---------------- #
def get_class_weights(labels):
    class_count = np.bincount(labels)
    weights = 1.0 / class_count
    weights = weights / weights.sum()
    return torch.tensor(weights, dtype=torch.float).to(device)

In [ ]:
from sklearn.metrics import roc_auc_score
from sklearn.preprocessing import label_binarize

def evaluate(model, loader, criterion):
    model.eval()
    all_preds, all_labels, all_probs = [], [], []
    total_loss = 0

    with torch.no_grad():
        for imgs, lbls in loader:
            imgs, lbls = imgs.to(device), lbls.to(device)

            outputs = model(imgs)
            loss = criterion(outputs, lbls)
            total_loss += loss.item()

            probs = torch.softmax(outputs, dim=1)
            preds = probs.argmax(dim=1)

            all_probs.extend(probs.cpu().numpy())
            all_preds.extend(preds.cpu().numpy())
            all_labels.extend(lbls.cpu().numpy())

    acc = accuracy_score(all_labels, all_preds)
    prec = precision_score(all_labels, all_preds, average="macro", zero_division=0)
    rec = recall_score(all_labels, all_preds, average="macro", zero_division=0)
    f1 = f1_score(all_labels, all_preds, average="macro", zero_division=0)

    # ROC-AUC
    y_true_bin = label_binarize(all_labels, classes=list(range(CONFIG["num_classes"])))
    roc_auc = roc_auc_score(y_true_bin, all_probs, average="macro", multi_class="ovr")

    cm = confusion_matrix(all_labels, all_preds)
    report = classification_report(all_labels, all_preds, zero_division=0)

    return acc, prec, rec, f1, roc_auc, total_loss / len(loader), cm, report, all_labels, all_preds


In [ ]:

def plot_cm(cm, class_names, title):
    plt.figure(figsize=(6, 5))
    sns.heatmap(cm, annot=True, fmt="d", cmap="Blues",
                xticklabels=class_names, yticklabels=class_names)
    plt.xlabel("Predicted")
    plt.ylabel("True")
    plt.title(title)

    img_path = f"{title.replace(' ', '_')}.png"
    plt.savefig(img_path, dpi=300, bbox_inches="tight")
    plt.close()
    return img_path


def train_fold(fold, train_idx, val_idx, class_names, group_name):
    if CONFIG.get("usewandb"):
        wandb.init(
            project=f"MobileNetV2_KFold_04-12-2025",
            name=f"{CONFIG['group_name']}_Fold-{fold+1}",
            group=group_name,
            config=CONFIG,
        )
    else:
        print("W&B disabled; skipping run init for this fold.")

    # Dataset split
    train_ds = Subset(dataset, train_idx)
    val_ds = Subset(dataset, val_idx)

    train_loader = DataLoader(train_ds, batch_size=CONFIG["batch_size"], shuffle=True)
    val_loader = DataLoader(val_ds, batch_size=CONFIG["batch_size"], shuffle=False)

    model = get_model(CONFIG["num_classes"]).to(device)

    # Criterion
    class_weights = get_class_weights(np.array(labels)[train_idx])
    criterion = nn.CrossEntropyLoss(weight=class_weights)

    optimizer = torch.optim.Adam(model.parameters(), lr=CONFIG["lr"])

    best_acc = 0
    best_prec = 0
    best_rec = 0
    best_rocauc = 0
    best_f1 = 0
    best_loss = float('inf')
    patience_counter = 0
    best_epoch = -1

    # For storing data of the best epoch
    best_val_labels, best_val_preds = None, None
    best_train_labels, best_train_preds = None, None
    best_val_cm, best_val_report = None, None # Initialize to None
    best_train_cm, best_train_report = None, None # Initialize to None

    for epoch in range(CONFIG["epochs"]):

        print(f"\n====== EPOCH {epoch+1}/{CONFIG['epochs']} ======")

        # ---------------- TRAIN ---------------- #
        model.train()
        train_loss_sum = 0
        all_train_preds, all_train_labels = [], [] # Accumulate predictions and labels for training metrics

        for imgs, lbls in tqdm(train_loader, desc="Training", leave=False):
            imgs, lbls = imgs.to(device), lbls.to(device)

            optimizer.zero_grad()
            outputs = model(imgs)
            loss = criterion(outputs, lbls)
            loss.backward()
            optimizer.step()

            train_loss_sum += loss.item()
            preds = outputs.argmax(1)
            all_train_preds.extend(preds.cpu().numpy())
            all_train_labels.extend(lbls.cpu().numpy())

        train_loss = train_loss_sum / len(train_loader)
        train_acc = accuracy_score(all_train_labels, all_train_preds)
        train_prec = precision_score(all_train_labels, all_train_preds, average="macro", zero_division=0)
        train_rec = recall_score(all_train_labels, all_train_preds, average="macro", zero_division=0)
        train_f1 = f1_score(all_train_labels, all_train_preds, average="macro", zero_division=0)

        # ---------------- VALIDATION ---------------- #
        val_acc, val_prec, val_rec, val_f1, val_rocauc, val_loss, \
val_cm, val_report, v_labels, v_preds = evaluate(model, val_loader, criterion)


        # WandB logs per epoch
        if CONFIG["usewandb"]:
            wandb.log({
                "epoch": epoch + 1,
                "train_loss": train_loss,
                "val_loss": val_loss,
                "train_accuracy": train_acc,
                "train_precision": train_prec,
                "train_recall": train_rec,
                "train_f1": train_f1,
                "val_accuracy": val_acc,
                "val_precision": val_prec,
                "val_recall": val_rec,
                "val_roc_auc": val_rocauc,
                "val_f1": val_f1,
            })
        print(f"Epoch {epoch+1}/{CONFIG['epochs']}")
        print(f"Train Accuracy: {train_acc:.4f} | Train Loss: {train_loss:.4f}")
        print(f"Test Accuracy: {val_acc:.4f} | Test Loss: {val_loss:.4f}")
        # ---------------- BEST EPOCH CHECK ---------------- #
        if val_acc > best_acc:
            best_acc = val_acc
            best_prec = val_prec
            best_rec = val_rec
            best_f1 = val_f1
            best_loss = val_loss
            best_rocauc = val_rocauc
            best_epoch = epoch + 1
            patience_counter = 0

            # Save model
            model_path = f"mobilenetnormal__fold{fold}.pth"
            torch.save(model.state_dict(), model_path)
            if CONFIG.get("usewandb"):
                wandb.save(model_path)
            else:
                print("W&B disabled; model saved locally only.")

            # Store validation info
            best_val_labels, best_val_preds = v_labels, v_preds
            best_val_cm, best_val_report = val_cm, val_report

            # Store training info for the best epoch
            best_train_labels, best_train_preds = all_train_labels, all_train_preds
            best_train_cm = confusion_matrix(best_train_labels, best_train_preds)
            best_train_report = classification_report(best_train_labels, best_train_preds, output_dict=False, zero_division=0)

        else:
            patience_counter += 1
            if patience_counter >= CONFIG["patience"]:
                print("EARLY STOPPING!")
                break

    # ---------------- LOGGING BEST EPOCH DATA ---------------- #

    print(f"\n===== BEST EPOCH = {best_epoch} | Best Val Acc = {best_acc:.4f} =====")

    # Generate heatmaps - Ensure best_val_cm and best_train_cm are not None
    if best_val_cm is not None and best_train_cm is not None:
        val_cm_img = plot_cm(best_val_cm, class_names, f"Fold{fold}_Validation_CM")
        train_cm_img = plot_cm(best_train_cm, class_names, f"Fold{fold}_Training_CM")
        if CONFIG.get("usewandb"):
            # Log confusion matrices as images
            wandb.log({
                "best_validation_confusion_matrix": wandb.Image(val_cm_img),
                "best_training_confusion_matrix": wandb.Image(train_cm_img),
                "best_validation_classification_report": wandb.Html(f"<pre>{best_val_report}</pre>"),
                "best_training_classification_report": wandb.Html(f"<pre>{best_train_report}</pre>")
            })
        else:
            print("W&B disabled; skipping confusion matrix & report logging.")
    else:
        print("Warning: No best model saved or metrics to log for best epoch.")

    if CONFIG.get("usewandb"):
        wandb.finish()
    else:
        print("W&B disabled; no run to finish.")

    return best_acc, best_prec, best_rec, best_f1, best_rocauc, best_loss



In [ ]:
# ---------------- MAIN ---------------- #
import pandas as pd
if __name__ == "__main__":

    if 'run' not in globals():
        run = 1

    # Preprocessing (ImageNet normalization)
    transform = transforms.Compose([
        transforms.ToTensor(),
        transforms.Normalize(
            mean=[0.485, 0.456, 0.406],
            std=[0.229, 0.224, 0.225]
        )
    ])

    # Load dataset
    files, labels = load_dataset(CONFIG["dataset_path"])
    class_names = sorted(os.listdir(CONFIG["dataset_path"]))
    dataset = PestDataset(files, labels, transform)

    # K-Fold setup
    kf = KFold(n_splits=CONFIG["k_folds"], shuffle=True, random_state=42)

    all_results = []

    for fold, (train_idx, val_idx) in enumerate(kf.split(files)):
        print(f"\n========== FOLD {fold+1} ==========")

        group_name = f"MobileNetV2_KFold_{CONFIG['group_name']}-{run}"

        # FIXED: Unpack 6 values (added best_rocauc)
        best_acc, best_prec, best_rec, best_f1, best_rocauc, best_loss = \
            train_fold(fold, train_idx, val_idx, class_names, group_name)

        all_results.append([fold+1, best_acc, best_prec, best_rec, best_f1, best_rocauc, best_loss])

    run += 1

    # Final summary table
    df = pd.DataFrame(all_results,
                      columns=["Fold", "Accuracy", "Precision", "Recall", "F1", "ROC-AUC", "Loss"])

    print("\n========== FINAL K-FOLD SUMMARY ==========")
    print(df.to_string(index=False))

    # Mean ± Std (for paper/report)
    print("\n========== MEAN ± STD ==========")
    for col in ["Accuracy", "Precision", "Recall", "F1", "ROC-AUC", "Loss"]:
        mean = df[col].mean()
        std = df[col].std()
        print(f"{col}: {mean:.4f} ± {std:.4f}")

    # Optional: Log summary table to wandb
    if CONFIG.get("usewandb"):
        wandb.init(
            project=f"MobileNetV2_KFold_04-12-2025",
            name=f"Summary_{CONFIG['group_name']}-{run}",
            job_type="summary"
        )

        # Log the summary table
        summary_table = wandb.Table(dataframe=df)
        wandb.log({"kfold_summary": summary_table})

        # Log mean metrics
        for col in ["Accuracy", "Precision", "Recall", "F1", "ROC-AUC", "Loss"]:
            wandb.log({
                f"mean_{col.lower()}": df[col].mean(),
                f"std_{col.lower()}": df[col].std()
            })

        wandb.finish()
    else:
        print("W&B disabled; skipping K-Fold summary upload.")


========== FOLD 1 ==========


epoch,▁▁▂▃▃▃▄▅▅▅▆▇▇▇█
train_accuracy,▁▇██▇█▇████████
train_f1,▁▇██▇█▇████████
train_loss,█▃▂▂▂▂▂▁▁▁▁▁▁▁▁
train_precision,▁▆█▇▇▇▇███▇████
train_recall,▁▆██▇█▇████████
val_accuracy,▆▁▃▃▆█▆▃▁▃▆▆██▆
val_f1,▆▁▃▃▆█▆▃▁▃▆▆██▆
val_loss,▂▁▂▁▁▄▄▃▃█▆▆▅▅▅
val_precision,▄▁▄▂▅█▅▄▁▄▇▇██▇
+2,...


/usr/local/lib/python3.12/dist-packages/torchvision/models/_utils.py:208: UserWarning: The parameter 'pretrained' is deprecated since 0.13 and may be removed in the future, please use 'weights' instead.
  warnings.warn(
/usr/local/lib/python3.12/dist-packages/torchvision/models/_utils.py:223: UserWarning: Arguments other than a weight enum or `None` for 'weights' are deprecated since 0.13 and may be removed in the future. The current behavior is equivalent to passing `weights=MobileNet_V2_Weights.IMAGENET1K_V1`. You can also use `weights=MobileNet_V2_Weights.DEFAULT` to get the most up-to-date weights.
  warnings.warn(msg)



====== EPOCH 1/50 ======


wandb: WARNING Symlinked 1 file into the W&B run directory; call wandb.save again to sync new files.


...

Epoch 20/50
Train Accuracy: 0.9937 | Train Loss: 0.0192
Val Accuracy: 0.9697 | Val Loss: 0.0581

====== EPOCH 21/50 ======


Epoch 21/50
Train Accuracy: 0.9887 | Train Loss: 0.0296
Val Accuracy: 0.9747 | Val Loss: 0.0656
EARLY STOPPING!

===== BEST EPOCH = 12 | Best Val Acc = 0.9798 =====


epoch,▁▁▂▂▂▃▃▃▄▄▅▅▅▆▆▆▇▇▇██
train_accuracy,▁▇▇▇█▇▇█████████████▇
train_f1,▁▇▇██▇▇██████████████
train_loss,█▂▂▂▁▂▂▁▁▁▁▁▁▁▁▁▁▁▁▁▁
train_precision,▁▇▇██▇▇███▇██████████
train_recall,▁▇▇██▇▇██████████████
val_accuracy,▁▆▃▇▇▅▅▆▇▆▇█▇▇▇▇▇▆▆▇▇
val_f1,▁▆▃▇▇▅▄▆▇▆▇█▇▇▇▇▇▆▆▇▇
val_loss,█▂█▃▂▃▆▂▁▄▂▁▁▂▂▂▂▃▃▁▂
val_precision,▁▅▃▇▆▅▄▅▆▆▆█▇▆▇▆▆▆▅▆▇
+2,...



========== FINAL K-FOLD SUMMARY ==========
 Fold  Accuracy  Precision   Recall       F1  ROC-AUC     Loss
    1  0.989950   0.990338 0.984496 0.987162 0.997368 0.119245
    2  0.994949   0.993827 0.989899 0.991757 0.999668 0.054758
    3  0.989899   0.985897 0.985897 0.985897 0.998778 0.062583
    4  0.984848   0.980490 0.983165 0.981771 0.992627 0.094618
    5  0.979798   0.975298 0.965686 0.970136 0.998374 0.057130

========== MEAN ± STD ==========
Accuracy: 0.9879 ± 0.0058
Precision: 0.9852 ± 0.0074
Recall: 0.9818 ± 0.0094
F1: 0.9833 ± 0.0082
ROC-AUC: 0.9974 ± 0.0028
Loss: 0.0777 ± 0.0282


mean_accuracy,▁
mean_f1,▁
mean_loss,▁
mean_precision,▁
mean_recall,▁
mean_roc-auc,▁
std_accuracy,▁
std_f1,▁
std_loss,▁
std_precision,▁
+2,...
